In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/acts_val.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_train.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/sc_val.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/test.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_train_f.bin
/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tribunals_val_f.bin
/kaggle/input/datasets/achiver07/data-project/acts_train.jsonl
/kaggle/input/datasets/achiver07/data-project/acts_val.jsonl
/kaggle/input/datasets/achiver07/data-project/sc_train.jsonl
/kaggle/input/datasets/achiver07/data-project/sc_val.jsonl
/kaggle/input/datasets/achiver07/data-project/test.json
/kaggle/input/datasets/achiver07/data-project/tribunals_train.jsonl
/kaggle/input/datasets/achiver07/data-pro

In [2]:
import os
import json
from collections import Counter
import numpy as np
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

# 1. Fetch Hugging Face token & download Phase_5_LSI
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")

path = snapshot_download(
    repo_id="Exploration-Lab/CS-781-Capstone",
    repo_type="dataset",
    token=HF_TOKEN,
    allow_patterns=["Phase_5_LSI/**"],
    local_dir="/kaggle/working",
)

# 2. Auto-locate train.json, dev.json, and test.json inside Phase_5_LSI
base_dir = "/kaggle/working/Phase_5_LSI"
found_files = {}
for root, _, files in os.walk(base_dir):
    for f in files:
        if f.endswith(".json"):
            found_files[f] = os.path.join(root, f)

print("Discovered JSON files:", found_files)

with open(found_files["train.json"], "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(found_files["dev.json"], "r", encoding="utf-8") as f:
    dev_data = json.load(f)
with open(found_files["test.json"], "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"\nCounts -> Train: {len(train_data)} | Dev: {len(dev_data)} | Test: {len(test_data)}")
print("Sample Train item keys:", list(train_data[0].keys()) if isinstance(train_data, list) else type(train_data))

if isinstance(train_data, list):
    sample = train_data[0]
    for k, v in sample.items():
        print(f"  {k} ({type(v).__name__}): {str(v)[:120]}")
    print("\nSample Test item:", {k: str(v)[:100] for k, v in test_data[0].items()})

    # Inspect label distribution if 'labels' key exists
    label_key = "labels" if "labels" in sample else [k for k in sample.keys() if k not in ("id", "text", "document")][0]
    all_train_labels = [lbl for ex in train_data for lbl in ex[label_key]]
    all_dev_labels = [lbl for ex in dev_data for lbl in ex[label_key]]
    counts = Counter(all_train_labels)
    unique_labels = sorted(set(all_train_labels).union(set(all_dev_labels)))

    print(f"\n=== Statute Label Statistics (Key: {label_key!r}) ===")
    print(f"Total unique statutes across Train+Dev: {len(unique_labels)} (In Train: {len(counts)}, In Dev: {len(set(all_dev_labels))})")
    print(f"Label ID range: min={min(unique_labels)}, max={max(unique_labels)}, type={type(unique_labels[0]).__name__}")
    print(f"Statute frequency in Train: min={min(counts.values())}, median={int(np.median(list(counts.values())))}, max={max(counts.values())}")
    print(f"Labels per document (Train): mean={np.mean([len(ex[label_key]) for ex in train_data]):.2f}, max={max(len(ex[label_key]) for ex in train_data)}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Discovered JSON files: {'test.json': '/kaggle/working/Phase_5_LSI/Test/test.json', 'dev.json': '/kaggle/working/Phase_5_LSI/Train/dev.json', 'train.json': '/kaggle/working/Phase_5_LSI/Train/train.json'}

Counts -> Train: 42750 | Dev: 10181 | Test: 13019
Sample Train item keys: ['id', 'text', 'labels']
  id (str): 1000008
  text (str): (a), <SECTION> of the <ACT>. The gist of theprosecution case relevant <ENTITY> the purpose of this proceedingmay be stat
  labels (list): [69, 9, 3]

Sample Test item: {'id': '100002997', 'text': '2.Prosecution case in brief was that <ENTITY>, daughter of informant <ENTITY> (PW-1) was married to '}

=== Statute Label Statistics (Key: 'labels') ===
Total unique statutes across Train+Dev: 100 (In Train: 100, In Dev: 100)
Label ID range: min=0, max=99, type=int
Statute frequency in Train: min=272, median=679, max=11731
Labels per document (Train): mean=3.71, max=31


In [3]:
import os
import gc
import json
import time
import math
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from sklearn.metrics import f1_score, accuracy_score
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

# =====================================================================
# 1. Download Dataset (If Needed) & Load Splits (100 Statutes: 0..99)
# =====================================================================
NUM_CLASSES = 100
TRAIN_PATH = "/kaggle/working/Phase_5_LSI/Train/train.json"
DEV_PATH   = "/kaggle/working/Phase_5_LSI/Train/dev.json"
TEST_PATH  = "/kaggle/working/Phase_5_LSI/Test/test.json"

if not os.path.exists(TRAIN_PATH):
    print("Downloading Phase_5_LSI from Hugging Face...")
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    snapshot_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        repo_type="dataset",
        token=HF_TOKEN,
        allow_patterns=["Phase_5_LSI/**"],
        local_dir="/kaggle/working",
    )

TOKENIZER_PATH = "/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1/tokenizer.json"
CKPT_PATHS = {
    "arch1": "/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt",
    "arch2": "/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt",
    "arch3": "/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/best.pt",
    "arch4": "/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt",
}

if "train_data" not in globals() or len(train_data) != 42750:
    print("Loading train.json, dev.json, and test.json...")
    with open(TRAIN_PATH, "r", encoding="utf-8") as f: train_data = json.load(f)
    with open(DEV_PATH, "r", encoding="utf-8") as f:   dev_data   = json.load(f)
    with open(TEST_PATH, "r", encoding="utf-8") as f:  test_data  = json.load(f)

def build_multihot(data, num_classes=100):
    Y = np.zeros((len(data), num_classes), dtype=np.float32)
    for i, ex in enumerate(data):
        for lbl in ex["labels"]:
            if 0 <= int(lbl) < num_classes:
                Y[i, int(lbl)] = 1.0
    return Y

Y_train = build_multihot(train_data, NUM_CLASSES)
Y_dev   = build_multihot(dev_data, NUM_CLASSES)

# Dampened inverse-frequency positive weights for rare statutes
pos_counts = Y_train.sum(axis=0).clip(min=1.0)
neg_counts = len(Y_train) - pos_counts
pos_weights_np = np.clip((neg_counts / pos_counts) ** 0.35, 1.0, 4.5).astype(np.float32)
pos_weights_tensor = torch.tensor(pos_weights_np, dtype=torch.float32, device="cuda")

# =====================================================================
# 2. Vectorized Per-Class Threshold Optimizer & Evaluation Metrics
# =====================================================================
def optimize_per_class_thresholds(dev_probs: np.ndarray, dev_targets: np.ndarray) -> np.ndarray:
    """Vectorized grid search over 86 thresholds in [0.10, 0.95] for each of the 100 statutes."""
    best_th = np.full(dev_probs.shape[1], 0.50, dtype=np.float32)
    best_f1 = np.full(dev_probs.shape[1], -1.0, dtype=np.float32)
    y_true = (dev_targets == 1.0)
    y_false = ~y_true

    for th in np.linspace(0.10, 0.95, 86):
        y_hat = (dev_probs >= th)
        tp = np.sum(y_hat & y_true, axis=0)
        fp = np.sum(y_hat & y_false, axis=0)
        fn = np.sum((~y_hat) & y_true, axis=0)
        f1_vec = (2.0 * tp) / (2.0 * tp + fp + fn + 1e-8)
        improved = f1_vec > best_f1
        best_f1[improved] = f1_vec[improved]
        best_th[improved] = th
    return best_th

def probs_to_binary_preds(probs: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    """Applies per-statute thresholds and guarantees at least 1 valid statute ID per case."""
    preds = (probs >= thresholds[None, :]).astype(np.int32)
    empty_rows = np.where(preds.sum(axis=1) == 0)[0]
    if len(empty_rows) > 0:
        top1_cols = np.argmax(probs[empty_rows] - thresholds[None, :], axis=1)
        preds[empty_rows, top1_cols] = 1
    return preds

def evaluate_lsi(preds_bin: np.ndarray, targets_bin: np.ndarray):
    macro_f1 = f1_score(targets_bin, preds_bin, average="macro", zero_division=0) * 100.0
    micro_f1 = f1_score(targets_bin, preds_bin, average="micro", zero_division=0) * 100.0
    subset_acc = accuracy_score(targets_bin, preds_bin) * 100.0
    return {
        "Macro-F1": round(macro_f1, 2),
        "Micro-F1": round(micro_f1, 2),
        "Subset-Acc": round(subset_acc, 2),
    }

def package_lsi_submission(test_examples, preds_bin: np.ndarray, zip_path: str):
    """Writes [{'id': ..., 'pred': [int, ...]}] to prediction.json at root of zip_path."""
    output_list = []
    for i, ex in enumerate(test_examples):
        statute_ids = sorted(list(set(int(c) for c in np.where(preds_bin[i] == 1)[0])))
        assert len(statute_ids) >= 1, f"Empty prediction for test doc {ex['id']}"
        output_list.append({"id": str(ex["id"]), "pred": statute_ids})

    assert len(output_list) == len(test_examples), "Test count mismatch!"
    tmp_json = "/kaggle/working/prediction.json"
    with open(tmp_json, "w", encoding="utf-8") as f:
        json.dump(output_list, f, indent=2)

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(tmp_json, arcname="prediction.json")

    with zipfile.ZipFile(zip_path, "r") as zf:
        assert zf.namelist() == ["prediction.json"]
    print(f"✅ Packaged {zip_path} ({os.path.getsize(zip_path)/1024:.1f} KB) -> ['prediction.json'] ({len(output_list)} test docs)")

# =====================================================================
# 3. Fast Pre-Tokenization (1024 Full-Doc Tokens + 256 Head+Tail Window)
# =====================================================================
tok = Tokenizer.from_file(TOKENIZER_PATH)
FULL_DOC_LEN = 768
WIN_SEQ_LEN  = 224

def pretokenize_dual_view(data_list, full_len=FULL_DOC_LEN, win_len=WIN_SEQ_LEN, batch_size=2048):
    n = len(data_list)
    arr_full = np.zeros((n, full_len), dtype=np.int32)
    arr_win  = np.zeros((n, win_len), dtype=np.int32)
    for i in range(0, n, batch_size):
        texts = []
        for ex in data_list[i:i+batch_size]:
            t = ex["text"]
            if len(t) > 6000:
                t = t[:3000] + " " + t[-3000:]
            texts.append(t)
        encs = tok.encode_batch(texts, add_special_tokens=False)
        for j, e in enumerate(encs):
            ids = e.ids
            if not ids:
                ids = [1]
            # 1. Full-Document View (up to 768 tokens: 50% Head + 50% Tail)
            if len(ids) >= full_len:
                hf = full_len // 2
                arr_full[i + j] = ids[:hf] + ids[-(full_len - hf):]
            else:
                arr_full[i + j, :len(ids)] = ids
            # 2. Transformer Context Window (224 tokens: 50% Head + 50% Tail)
            if len(ids) >= win_len:
                hw = win_len // 2
                arr_win[i + j] = ids[:hw] + ids[-(win_len - hw):]
            else:
                arr_win[i + j, :len(ids)] = ids
    return arr_win, arr_full

if "X_train_win" not in globals():
    print("\nPre-tokenizing Train (42,750), Dev (10,181), and Test (13,019) into Dual-View Arrays...")
    t_p = time.time()
    X_train_win, X_train_full = pretokenize_dual_view(train_data)
    X_dev_win,   X_dev_full   = pretokenize_dual_view(dev_data)
    X_test_win,  X_test_full  = pretokenize_dual_view(test_data)
    print(f"Dual-view pre-tokenization completed in {time.time()-t_p:.1f}s!")
else:
    print("\nReusing already pre-tokenized Dual-View Arrays in memory!")

class DualViewLSIDataset(Dataset):
    def __init__(self, X_win, X_full, Y_labels=None):
        self.X_win = X_win
        self.X_full = X_full
        self.Y = Y_labels
    def __len__(self):
        return len(self.X_win)
    def __getitem__(self, idx):
        w = torch.from_numpy(self.X_win[idx]).long()
        f = torch.from_numpy(self.X_full[idx]).long()
        if self.Y is not None:
            return w, f, torch.from_numpy(self.Y[idx]).float()
        return w, f

# =====================================================================
# 4. Reconstruct All 4 Phase 1 Architectures + Approved Sec 11.2.1 PEFT
# =====================================================================
def apply_rope(x, cos, sin):
    d = x.shape[-1] // 2
    return torch.cat([x[..., :d] * cos - x[..., d:] * sin, x[..., d:] * cos + x[..., :d] * sin], dim=-1)

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
    def forward(self, seq_len, device, dtype):
        t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        return freqs.cos().to(dtype)[None, None, :, :], freqs.sin().to(dtype)[None, None, :, :]

# --- Arch 1 (8L Decoder, d=768) ---
class Arch1Attention(nn.Module):
    def __init__(self, d_model=768, n_heads=12, max_len=1024):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.register_buffer("mask", torch.tril(torch.ones(1, 1, max_len, max_len)))
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
    def forward(self, x):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, C))

class Arch1MLP(nn.Module):
    def __init__(self, d_model=768, d_ff=3072):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
    def forward(self, x): return self.fc2(F.gelu(self.fc1(x)))

class Arch1Block(nn.Module):
    def __init__(self, d_model=768, n_heads=12, d_ff=3072):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = Arch1Attention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = Arch1MLP(d_model, d_ff)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))

class Arch1Backbone(nn.Module):
    def __init__(self, vocab_size=40000, d_model=768, n_layers=8):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(1024, d_model)
        self.blocks = nn.ModuleList([Arch1Block(d_model) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
    def get_embedding_weight(self):
        return self.token_emb.weight
    def forward(self, input_ids):
        B, T = input_ids.size()
        pos = torch.arange(0, T, device=input_ids.device).clamp(max=1023)
        x = self.token_emb(input_ids) + self.pos_emb(pos)
        for blk in self.blocks: x = blk(x)
        return self.final_norm(x)

# --- Arch 2 (6L Bidirectional Encoder from Seq2Seq, d=768) ---
class Arch2Attention(nn.Module):
    def __init__(self, d_model=768, n_heads=12):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.rope = RotaryEmbedding(self.head_dim)
    def forward(self, x):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        cos, sin = self.rope(T, x.device, x.dtype)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, C))

class Arch2FFN(nn.Module):
    def __init__(self, d_model=768, d_ff=3072):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
    def forward(self, x): return self.linear2(F.gelu(self.linear1(x)))

class Arch2EncLayer(nn.Module):
    def __init__(self, d_model=768):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn = Arch2Attention(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = Arch2FFN(d_model)
    def forward(self, x):
        x = x + self.self_attn(self.norm1(x))
        return x + self.ffn(self.norm2(x))

class Arch2EncoderBackbone(nn.Module):
    def __init__(self, vocab_size=40000, d_model=768, n_layers=6):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.encoder = nn.Module()
        self.encoder.layers = nn.ModuleList([Arch2EncLayer(d_model) for _ in range(n_layers)])
    def get_embedding_weight(self):
        return self.token_embedding.weight
    def forward(self, input_ids):
        x = self.token_embedding(input_ids)
        for layer in self.encoder.layers: x = layer(x)
        return x

# --- Arch 3 (16L SWA, d=1024) ---
class Arch3Block(nn.Module):
    def __init__(self, d_model=1024, n_heads=16, d_ff=4096):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = Arch2Attention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = Arch1MLP(d_model, d_ff)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))

class Arch3Backbone(nn.Module):
    def __init__(self, vocab_size=40000, d_model=1024, n_layers=16):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([Arch3Block(d_model) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
    def get_embedding_weight(self):
        return self.token_emb.weight
    def forward(self, input_ids):
        x = self.token_emb(input_ids)
        for blk in self.blocks: x = blk(x)
        return self.final_norm(x)

# --- Arch 4 (12L Hybrid GQA, d=1024) ---
class Arch4GQAAttention(nn.Module):
    def __init__(self, d_model=1024, n_heads=16, n_kv_heads=8):
        super().__init__()
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.head_dim = d_model // n_heads
        self.kv_dim = n_kv_heads * self.head_dim
        self.qkv_proj = nn.Linear(d_model, d_model + 2 * self.kv_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rope = RotaryEmbedding(self.head_dim)
    def forward(self, x):
        B, T, C = x.size()
        q, k, v = torch.split(self.qkv_proj(x), [C, self.kv_dim, self.kv_dim], dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        cos, sin = self.rope(T, x.device, x.dtype)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        k = k.repeat_interleave(self.n_heads // self.n_kv_heads, dim=1)
        v = v.repeat_interleave(self.n_heads // self.n_kv_heads, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, C))

class Arch4GatedFFN(nn.Module):
    def __init__(self, d_model=1024, d_ff=3328):
        super().__init__()
        self.in_proj = nn.Linear(d_model, 2 * d_ff, bias=False)
        self.out_proj = nn.Linear(d_ff, d_model, bias=False)
    def forward(self, x):
        gate, up = self.in_proj(x).chunk(2, dim=-1)
        return self.out_proj(F.silu(gate) * up)

class Arch4Layer(nn.Module):
    def __init__(self, d_model=1024):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = Arch4GQAAttention(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = Arch4GatedFFN(d_model)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.ffn(self.norm2(x))

class Arch4Backbone(nn.Module):
    def __init__(self, vocab_size=40000, d_model=1024, n_layers=12):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([Arch4Layer(d_model) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
    def get_embedding_weight(self):
        return self.token_embedding.weight
    def forward(self, input_ids):
        x = self.token_embedding(input_ids)
        for layer in self.layers: x = layer(x)
        return self.final_norm(x)

# --- Approved Section 11.2.1 PEFT Modules: LORMA & AdaLoRA ---
class LORMALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, rank: int = 16, alpha: float = 32.0):
        super().__init__()
        self.base_layer = base_layer
        for p in self.base_layer.parameters(): p.requires_grad = False
        self.scaling = alpha / rank
        self.lorma_A = nn.Parameter(torch.empty(rank, base_layer.out_features))
        self.lorma_B = nn.Parameter(torch.zeros(base_layer.out_features, rank))
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))
    def forward(self, x):
        b = self.base_layer(x)
        return b + ((b @ self.lorma_A.T) @ self.lorma_B.T) * self.scaling

class AdaLoRALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, rank: int = 16, alpha: float = 32.0):
        super().__init__()
        self.base_layer = base_layer
        for p in self.base_layer.parameters(): p.requires_grad = False
        self.scaling = alpha / rank
        self.P = nn.Parameter(torch.empty(base_layer.out_features, rank))
        self.Lambda = nn.Parameter(torch.zeros(rank))
        self.Q = nn.Parameter(torch.empty(rank, base_layer.in_features))
        nn.init.normal_(self.P, std=0.02)
        nn.init.normal_(self.Q, std=0.02)
    def forward(self, x):
        return self.base_layer(x) + (((x @ self.Q.T) * self.Lambda) @ self.P.T) * self.scaling

# =====================================================================
# 5. Hierarchical Dual-View Multi-Label Classifier (Transformer + Full-Doc Phase 1 Emb)
# =====================================================================
class HierarchicalPhase1Classifier(nn.Module):
    """
    Combines:
      1. Deep Contextual Transformer Pooling (Mean + Max + Attention) over the 224-token Head+Tail window
      2. Full-Document Hierarchical Pooling (768 tokens) initialized from Phase 1 Pretrained Token Embeddings
    """
    def __init__(self, backbone: nn.Module, d_model: int, num_classes: int = 100):
        super().__init__()
        self.backbone = backbone
        self.attn_gate = nn.Linear(d_model, 1, bias=False)

        # Initialize Full-Document Embedding Branch directly from Phase 1 Pretrained Token Embeddings
        pretrained_emb = backbone.get_embedding_weight().detach().clone()
        self.full_doc_emb = nn.Embedding.from_pretrained(pretrained_emb, freeze=False, padding_idx=0)
        self.full_doc_attn = nn.Linear(d_model, 1, bias=False)

        self.deep_head = nn.Sequential(
            nn.LayerNorm(d_model * 3),
            nn.Dropout(0.15),
            nn.Linear(d_model * 3, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
            nn.Linear(d_model, num_classes)
        )
        self.wide_head = nn.Sequential(
            nn.LayerNorm(d_model * 2),
            nn.Dropout(0.15),
            nn.Linear(d_model * 2, d_model // 2),
            nn.GELU(),
            nn.LayerNorm(d_model // 2),
            nn.Linear(d_model // 2, num_classes)
        )

    def forward(self, x_win, x_full):
        # 1. Deep Contextual Transformer Branch (x_win: [B, 224])
        mask_w = (x_win != 0).unsqueeze(-1).float()
        h = self.backbone(x_win)
        mean_w = (h * mask_w).sum(dim=1) / mask_w.sum(dim=1).clamp(min=1e-6)
        max_w  = h.masked_fill(mask_w == 0, -1e4).max(dim=1).values
        sc_w   = self.attn_gate(h).squeeze(-1).masked_fill(x_win == 0, -1e4)
        attn_w = (h * F.softmax(sc_w, dim=-1).unsqueeze(-1)).sum(dim=1)
        deep_logits = self.deep_head(torch.cat([mean_w, max_w, attn_w], dim=-1))

        # 2. Full-Document Phase 1 Embedding Branch (x_full: [B, 768])
        mask_f = (x_full != 0).unsqueeze(-1).float()
        e_full = self.full_doc_emb(x_full)
        mean_f = (e_full * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1e-6)
        sc_f   = self.full_doc_attn(e_full).squeeze(-1).masked_fill(x_full == 0, -1e4)
        attn_f = (e_full * F.softmax(sc_f, dim=-1).unsqueeze(-1)).sum(dim=1)
        wide_logits = self.wide_head(torch.cat([mean_f, attn_f], dim=-1))

        return deep_logits + wide_logits

# =====================================================================
# 6. Fast Training, Safe Autocast KD & Inference Engine (float16)
# =====================================================================
@torch.no_grad()
def predict_logits_dual(model, X_win, X_full, batch_size=128):
    model.eval()
    loader = DataLoader(DualViewLSIDataset(X_win, X_full), batch_size=batch_size, shuffle=False)
    all_logits = []
    for bw, bf in loader:
        with torch.amp.autocast('cuda', dtype=torch.float16):
            logits = model(bw.cuda(), bf.cuda())
        all_logits.append(logits.float().cpu().numpy())
    return np.concatenate(all_logits, axis=0)

def run_lsi_experiment(
    exp_key: str,
    exp_label: str,
    arch_name: str,
    method_name: str,
    teacher_model: nn.Module = None,
    max_steps: int = 550,
    batch_size: int = 64,
    lr: float = 8e-4,
):
    print(f"\n=======================================================")
    print(f"🚀 Running {exp_label}: {arch_name.upper()} + {method_name}")
    print(f"=======================================================")
    t0 = time.time()
    ckpt = torch.load(CKPT_PATHS[arch_name], map_location="cpu", weights_only=False)
    sd = {k.replace("_orig_mod.", ""): v for k, v in ckpt["model"].items()}

    if arch_name == "arch4":
        bb, d_model = Arch4Backbone(), 1024
    elif arch_name == "arch3":
        bb, d_model = Arch3Backbone(), 1024
    elif arch_name == "arch2":
        bb, d_model = Arch2EncoderBackbone(), 768
    elif arch_name == "arch1":
        bb, d_model = Arch1Backbone(), 768
    bb.load_state_dict(sd, strict=False)
    del ckpt, sd; gc.collect()

    for p in bb.parameters():
        p.requires_grad = False

    if method_name == "LORMA":
        for name, mod in list(bb.named_modules()):
            if isinstance(mod, nn.Linear):
                p_n, c_n = name.rsplit(".", 1)
                setattr(bb.get_submodule(p_n), c_n, LORMALinear(mod, rank=16))
        for m in bb.modules():
            if isinstance(m, nn.LayerNorm):
                for p in m.parameters(): p.requires_grad = True

    elif method_name == "Partial_Layerwise_FT":
        for layer in bb.encoder.layers[-2:]:
            for p in layer.parameters(): p.requires_grad = True

    elif method_name == "AdaLoRA":
        for name, mod in list(bb.named_modules()):
            if isinstance(mod, nn.Linear):
                p_n, c_n = name.rsplit(".", 1)
                setattr(bb.get_submodule(p_n), c_n, AdaLoRALinear(mod, rank=16))
        for m in bb.modules():
            if isinstance(m, nn.LayerNorm):
                for p in m.parameters(): p.requires_grad = True

    elif method_name == "Knowledge_Distillation":
        for name, mod in list(bb.named_modules()):
            if isinstance(mod, nn.Linear):
                p_n, c_n = name.rsplit(".", 1)
                setattr(bb.get_submodule(p_n), c_n, LORMALinear(mod, rank=16))
        for p in bb.blocks[-1].parameters():
            p.requires_grad = True

    model = HierarchicalPhase1Classifier(bb, d_model=d_model, num_classes=NUM_CLASSES).cuda()
    tr_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot_p = sum(p.numel() for p in model.parameters())
    print(f"  Trainable params: {tr_p:,} / {tot_p:,} ({100*tr_p/tot_p:.2f}%)")

    train_loader = DataLoader(DualViewLSIDataset(X_train_win, X_train_full, Y_train), batch_size=batch_size, shuffle=True, drop_last=True)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_steps, eta_min=lr * 0.1)
    scaler = torch.amp.GradScaler('cuda')
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights_tensor)

    model.train()
    if teacher_model is not None:
        teacher_model.eval()

    step = 0
    running_loss = 0.0
    while step < max_steps:
        for bw, bf, by in train_loader:
            step += 1
            bw, bf, by = bw.cuda(), bf.cuda(), by.cuda()
            with torch.amp.autocast('cuda', dtype=torch.float16):
                logits = model(bw, bf)
                bce_loss = criterion(logits, by)

                if method_name == "Knowledge_Distillation" and teacher_model is not None:
                    with torch.no_grad():
                        t_logits = teacher_model(bw, bf)
                    # Autocast-safe Binary Cross-Entropy with Logits for Multi-Label KD
                    T = 2.0
                    p_t = torch.sigmoid(t_logits.detach() / T)
                    kd_loss = F.binary_cross_entropy_with_logits(logits / T, p_t, reduction="mean") * (T ** 2)
                    loss = 0.5 * bce_loss + 0.5 * kd_loss
                else:
                    loss = bce_loss

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            opt.zero_grad()
            running_loss += loss.item()

            if step % 150 == 0 or step == max_steps:
                print(f"  Step {step}/{max_steps} | Avg Loss: {running_loss/step:.4f} | Time: {time.time()-t0:.1f}s")
            if step >= max_steps:
                break

    print("  Predicting on Dev (10,181) & Test (13,019)...")
    dev_probs  = 1.0 / (1.0 + np.exp(-predict_logits_dual(model, X_dev_win, X_dev_full, batch_size=128)))
    test_probs = 1.0 / (1.0 + np.exp(-predict_logits_dual(model, X_test_win, X_test_full, batch_size=128)))

    best_th = optimize_per_class_thresholds(dev_probs, Y_dev)
    dev_preds_bin  = probs_to_binary_preds(dev_probs, best_th)
    test_preds_bin = probs_to_binary_preds(test_probs, best_th)

    metrics = evaluate_lsi(dev_preds_bin, Y_dev)
    print(f"  [{exp_label}] Dev -> Macro-F1: {metrics['Macro-F1']}% | Micro-F1: {metrics['Micro-F1']}% | Subset-Acc: {metrics['Subset-Acc']}%")

    package_lsi_submission(test_data, test_preds_bin, f"/kaggle/working/submission_{exp_key}.zip")
    return model, dev_probs, test_probs, metrics

# =====================================================================
# 7. Execute All 4 Required Experiments + Final Ensemble
# =====================================================================
summary_rows = []
dev_prob_bank = {}
test_prob_bank = {}

# 1. Exp 1: Arch 4 (242M Hybrid GQA) + Compulsory LORMA (Teacher for Exp 4)
teacher_arch4, d_p1, t_p1, m1 = run_lsi_experiment(
    exp_key="exp1_arch4_lorma",
    exp_label="Exp 1 (Teacher)",
    arch_name="arch4",
    method_name="LORMA",
    max_steps=550,
    batch_size=64,
    lr=8e-4
)
dev_prob_bank["exp1"]  = d_p1
test_prob_bank["exp1"] = t_p1
summary_rows.append(("Exp 1", "Arch 4 (242M GQA)", "LORMA (Compulsory)", m1["Macro-F1"], m1["Micro-F1"], m1["Subset-Acc"]))

# 2. Exp 4: Arch 1 (127M Decoder) + Mandatory Knowledge Distillation (From Teacher Arch 4)
student_arch1, d_p4, t_p4, m4 = run_lsi_experiment(
    exp_key="exp4_arch1_kd",
    exp_label="Exp 4 (KD)",
    arch_name="arch1",
    method_name="Knowledge_Distillation",
    teacher_model=teacher_arch4,
    max_steps=550,
    batch_size=64,
    lr=8e-4
)
dev_prob_bank["exp4"]  = d_p4
test_prob_bank["exp4"] = t_p4
summary_rows.append(("Exp 4 (KD)", "Arch 1 (127M Dec)", "Knowledge Distillation", m4["Macro-F1"], m4["Micro-F1"], m4["Subset-Acc"]))

del teacher_arch4, student_arch1
gc.collect()
torch.cuda.empty_cache()

# 3. Exp 2: Arch 2 (74M Bidirectional Encoder) + Partial/Layer-wise Fine-Tuning
model_arch2, d_p2, t_p2, m2 = run_lsi_experiment(
    exp_key="exp2_arch2_partial_ft",
    exp_label="Exp 2",
    arch_name="arch2",
    method_name="Partial_Layerwise_FT",
    max_steps=550,
    batch_size=64,
    lr=8e-4
)
dev_prob_bank["exp2"]  = d_p2
test_prob_bank["exp2"] = t_p2
summary_rows.append(("Exp 2", "Arch 2 (74M Enc)", "Partial Layerwise FT", m2["Macro-F1"], m2["Micro-F1"], m2["Subset-Acc"]))
del model_arch2; gc.collect(); torch.cuda.empty_cache()

# 4. Exp 3: Arch 3 (283M SWA) + AdaLoRA
model_arch3, d_p3, t_p3, m3 = run_lsi_experiment(
    exp_key="exp3_arch3_adalora",
    exp_label="Exp 3",
    arch_name="arch3",
    method_name="AdaLoRA",
    max_steps=500,
    batch_size=64,
    lr=8e-4
)
dev_prob_bank["exp3"]  = d_p3
test_prob_bank["exp3"] = t_p3
summary_rows.append(("Exp 3", "Arch 3 (283M SWA)", "AdaLoRA (SVD Rank)", m3["Macro-F1"], m3["Micro-F1"], m3["Subset-Acc"]))
del model_arch3; gc.collect(); torch.cuda.empty_cache()

# 5. Multi-Architecture Ensemble (submission.zip)
ens_dev_probs  = (0.30 * dev_prob_bank["exp1"] + 0.25 * dev_prob_bank["exp2"] + 0.20 * dev_prob_bank["exp3"] + 0.25 * dev_prob_bank["exp4"])
ens_test_probs = (0.30 * test_prob_bank["exp1"] + 0.25 * test_prob_bank["exp2"] + 0.20 * test_prob_bank["exp3"] + 0.25 * test_prob_bank["exp4"])
th_ens = optimize_per_class_thresholds(ens_dev_probs, Y_dev)
m_ens = evaluate_lsi(probs_to_binary_preds(ens_dev_probs, th_ens), Y_dev)
summary_rows.append(("Ensemble", "All 4 Phase 1 Archs", "Per-Statute Calibrated", m_ens["Macro-F1"], m_ens["Micro-F1"], m_ens["Subset-Acc"]))
package_lsi_submission(test_data, probs_to_binary_preds(ens_test_probs, th_ens), "/kaggle/working/submission.zip")

print("\n" + "=" * 95)
print(f"{'Exp':<12} | {'Phase 1 Model':<19} | {'Technique (Sec 11.2.1)':<24} | {'Macro-F1':<9} | {'Micro-F1':<9} | {'Subset-Acc':<10}")
print("-" * 95)
for r in summary_rows:
    print(f"{r[0]:<12} | {r[1]:<19} | {r[2]:<24} | {r[3]:<9.2f} | {r[4]:<9.2f} | {r[5]:<10.2f}")
print("=" * 95)


Pre-tokenizing Train (42,750), Dev (10,181), and Test (13,019) into Dual-View Arrays...
Dual-view pre-tokenization completed in 51.0s!

🚀 Running Exp 1 (Teacher): ARCH4 + LORMA
  Trainable params: 49,504,968 / 250,897,096 (19.73%)
  Step 150/550 | Avg Loss: 0.2746 | Time: 205.5s
  Step 300/550 | Avg Loss: 0.2511 | Time: 403.3s
  Step 450/550 | Avg Loss: 0.2379 | Time: 601.1s
  Step 550/550 | Avg Loss: 0.2318 | Time: 732.9s
  Predicting on Dev (10,181) & Test (13,019)...
  [Exp 1 (Teacher)] Dev -> Macro-F1: 35.05% | Micro-F1: 44.13% | Subset-Acc: 2.44%
✅ Packaged /kaggle/working/submission_exp1_arch4_lorma.zip (132.2 KB) -> ['prediction.json'] (13019 test docs)

🚀 Running Exp 4 (KD): ARCH1 + Knowledge_Distillation
  Trainable params: 42,064,712 / 123,187,784 (34.15%)
  Step 150/550 | Avg Loss: 0.9898 | Time: 161.8s
  Step 300/550 | Avg Loss: 0.9589 | Time: 312.0s
  Step 450/550 | Avg Loss: 0.9441 | Time: 463.0s
  Step 550/550 | Avg Loss: 0.9378 | Time: 562.5s
  Predicting on Dev (10,18

In [4]:
fgvgh

NameError: name 'fgvgh' is not defined